# Descriptive Statistics of the Corpus

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = 'data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

In [3]:
# to rename the corpus correctly . . . 
def lang(x):
    return x.split('-')[1]

In [4]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [5]:
fs = grab_all_files(DATA_PATH, '.tsv')
fs

['data/lme_ready/cha-croatian-ceda-analysis.tsv',
 'data/lme_ready/cha-yiddish-ceda-analysis.tsv',
 'data/lme_ready/cha-callfriend-ceda-analysis.tsv',
 'data/lme_ready/cha-callhome-ceda-analysis.tsv',
 'data/lme_ready/cha-finchat_corpus-ceda-analysis.tsv',
 'data/lme_ready/cha-ko_corpus-ceda-analysis.tsv']

In [13]:
final_tallies = dict()

In [14]:
turn_cut_off = 5

In [15]:
for f in tqdm(fs):
    stats = []
    print(f)
    df = pd.read_table(f,sep='\t')
    print(df['x_speaker'].unique()[0].split('-')[-4:-1])
    df['conversation_id'] = ['-'.join(uid.split('-')[-4:-1]) for uid in df['x_speaker']]
    
    # if ('/cha-' in f) or ('grace-' in f):
    #     df['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df['x_speaker'])]
    # 
    # if '/candor-' in f:
    #     df['conversation_id'] = [sp.replace('.csv', '') for sp in tqdm(df['conversation_id'])]
    #     df['corpus'] = 'CANDOR'
    
    # else:
    #     if 'conversation_id' in df.columns:
    #         0
    #     else:
    #         df['conversation_id'] = f
    
    df['x_turn_id_'] = df['conversation_id'] + '-' + df['x_turn_id'].astype(str)
    df['y_turn_id_'] = df['conversation_id'] + '-' + df['y_turn_id'].astype(str)
    df['lang'] = [lang(co) for co in tqdm(df['corpus'])]
    
    after_kth_turn = df['x_turn_id'] >= turn_cut_off
    appropriate_n_tokens = (df['nx'] >= 5) & (df['ny'] >= 5)
    
    for corpus in df['corpus'].unique():
        corpus_sel = df['corpus'].isin([corpus])
        
        if corpus in final_tallies.keys():
            
            ##### all data
            final_tallies[corpus]['all data']['files'] += [f]
            final_tallies[corpus]['all data']['speakers'] = np.concatenate([
                final_tallies[corpus]['all data']['speakers'],
                np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel].values)
            ])
            final_tallies[corpus]['all data']['conversations'] = np.concatenate([
                final_tallies[corpus]['all data']['conversations'],
                df['conversation_id'].loc[corpus_sel].unique()
                
            ])
            final_tallies[corpus]['all data']['utterances'] += len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel].values))
            final_tallies[corpus]['all data']['comparisons'] += len(df.loc[corpus_sel])
            
            
            ##### after fifth turn
            final_tallies[corpus]['after fifth turn']['files'] += [f]
            final_tallies[corpus]['after fifth turn']['speakers'] = np.concatenate([
                final_tallies[corpus]['after fifth turn']['speakers'],
                np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn].values)
            ])
            final_tallies[corpus]['after fifth turn']['conversations'] = np.concatenate([
                final_tallies[corpus]['after fifth turn']['conversations'],
                df['conversation_id'].loc[corpus_sel & after_kth_turn].unique()
                
            ])
            final_tallies[corpus]['after fifth turn']['utterances'] += len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn].values))
            final_tallies[corpus]['after fifth turn']['comparisons'] += len(df.loc[corpus_sel & after_kth_turn])
            
            
            ##### fully parsed
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['files'] += [f]
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['speakers'] = np.concatenate([
                final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['speakers'],
                np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values)
            ])
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversations'] = np.concatenate([
                final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['conversations'],
                df['conversation_id'].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].unique()
                
            ])
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['utterances'] += len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values))
            final_tallies[corpus]['after fifth turn & $n_{tokens} \geq 5$']['comparisons'] += len(df.loc[corpus_sel & after_kth_turn & appropriate_n_tokens])
            
                
        else:
            final_tallies[corpus] = {
                # all data
                'all_data': {
                    'files': [f],
                    'split': 'all data',
                    'speakers': np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel].values),
                    'conversations': df['conversation_id'].loc[corpus_sel].unique(),
                    'utterances': len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel].values)),
                    'comparisons': len(df.loc[corpus_sel])
                },
                
                # after fifth turn
                'after fifth turn': {
                    'files': [f],
                    'split': 'after fifth turn',
                    'speakers': np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn].values),
                    'conversations': df['conversation_id'].loc[corpus_sel & after_kth_turn].unique(),
                    'utterances': len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn].values)),
                    'comparisons': len(df.loc[corpus_sel & after_kth_turn])
                },
                
                # after fifth turn & n >= 5
                'after fifth turn & $n_{tokens} \geq 5$': {
                    'files': [f],
                    'split': 'after fifth turn & $n_{tokens} \geq 5$',
                    'speakers': np.unique(df[['x_speaker', 'y_speaker']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values),
                    'conversations': df['conversation_id'].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].unique(),
                    'utterances': len(np.unique(df[['x_turn_id_', 'y_turn_id_']].loc[corpus_sel & after_kth_turn & appropriate_n_tokens].values)),
                    'comparisons': len(df.loc[corpus_sel & after_kth_turn & appropriate_n_tokens])
                }
            }
    
    del df

  0%|          | 0/6 [00:00<?, ?it/s]

data/lme_ready/cha-croatian-ceda-analysis.tsv
['croation', 'cro', '2011_56']


  0%|          | 0/3311067 [00:00<?, ?it/s]

data/lme_ready/cha-yiddish-ceda-analysis.tsv
['yid', 'men', '1']


  0%|          | 0/13121 [00:00<?, ?it/s]

...

  0%|          | 0/11122234 [00:00<?, ?it/s]

data/lme_ready/cha-finchat_corpus-ceda-analysis.tsv
['finchat', 'fin', '10']


  0%|          | 0/92605 [00:03<?, ?it/s]

data/lme_ready/cha-ko_corpus-ceda-analysis.tsv
['callfriend', 'ko', '4012']


  0%|          | 0/2197289 [00:00<?, ?it/s]

In [16]:
stats = []

In [17]:
for corpus, splits in final_tallies.items():
    for k,v in splits.items():
        stats += [v]
        stats[-1]['corpus'] = corpus

In [18]:
stats = pd.DataFrame(stats)

In [19]:
stats['speakers'] = [len(np.unique(sp)) for sp in stats['speakers']]
stats['conversations'] = [len(np.unique(sp)) for sp in stats['conversations']]

In [20]:
stats.head(n=100)

,files,split,speakers,conversations,utterances,comparisons,corpus
0,[data/lme_ready/cha-croatian-ceda-analysis.tsv],all data,595,164,120440,3311067,croation-cro
1,[data/lme_ready/cha-croatian-ceda-analysis.tsv],after fifth turn,594,164,119136,3271436,croation-cro
2,[data/lme_ready/cha-croatian-ceda-analysis.tsv],after fifth turn & $n_{tokens} \geq 5$,588,164,96865,2155034,croation-cro
3,[data/lme_ready/cha-yiddish-ceda-analysis.tsv],all data,17,11,534,11923,yiddish-yid-men
4,[data/lme_ready/cha-yiddish-ceda-analysis.tsv],after fifth turn,12,11,490,10825,yiddish-yid-men
5,[data/lme_ready/cha-yiddish-ceda-analysis.tsv],after fifth turn & $n_{tokens} \geq 5$,12,11,477,10322,yiddish-yid-men
6,[data/lme_ready/cha-yiddish-ceda-analysis.tsv],all data,18,16,168,1198,yiddish-yid-women
7,[data/lme_ready/cha-yiddish-ceda-analysis.tsv],after fifth turn,13,12,105,685,yiddish-yid-women
8,[data/lme_ready/cha-yiddish-ceda-analysis.tsv],after fifth turn & $n_{tokens} \geq 5$,13,12,105,685,yiddish-yid-women
9,[data/lme_ready/cha-callfriend-ceda-analysis.tsv],all data,70,31,18200,1064461,callfriend-eng-n


In [21]:
stats.to_csv(os.path.join(DATA_LOC,'corpus-descriptives.csv'), index=False, encoding='utf-8')